# WasteLens · TrashNet notebook

Run the setup and data audit before implementing the classifier. Use `class_id` as the label index, choose the checkpoint on `val`, and reserve `test` for final evaluation. On Kaggle, enable Internet so the setup can fetch the public repo and the pinned dataset.

In [ ]:
from pathlib import Path
import subprocess
import sys

kaggle_working = Path("/kaggle/working")
repo_candidates = [Path.cwd(), *Path.cwd().parents, kaggle_working / "wastelens"]
root = next((p for p in repo_candidates if (p / "scripts/prepare_trashnet.py").is_file()), None)
if root is None and kaggle_working.is_dir():
    root = kaggle_working / "wastelens"
    subprocess.run(
        ["git", "clone", "https://github.com/Quang-Luu-2005/wastelens.git", str(root)],
        check=True,
    )
if root is None:
    raise FileNotFoundError("Run this notebook from the WasteLens repo or open it on Kaggle.")

subprocess.run([sys.executable, str(root / "scripts/prepare_trashnet.py")], cwd=root, check=True);

In [ ]:
import csv
import json
from collections import Counter

with (root / "data/splits/trashnet.csv").open(encoding="utf-8", newline="") as stream:
    rows = list(csv.DictReader(stream))
summary = json.loads((root / "data/splits/trashnet-summary.json").read_text(encoding="utf-8"))
image_root = root / "data/trashnet"
rows_by_split = {name: [row for row in rows if row["split"] == name] for name in ("train", "val", "test")}
class_names = [label for _, label in sorted({(int(row["class_id"]), row["label"]) for row in rows})]

assert len({row["path"] for row in rows}) == len(rows) == summary["retained_images"]
assert {name: len(items) for name, items in rows_by_split.items()} == summary["split_counts"]
assert all((image_root / row["path"]).is_file() for row in rows)

print(f"{'class':<12}{'train':>6}{'val':>6}{'test':>6}")
for label in class_names:
    counts = Counter(row["split"] for row in rows if row["label"] == label)
    print(f"{label:<12}{counts['train']:>6}{counts['val']:>6}{counts['test']:>6}")

In [ ]:
from IPython.display import display
from PIL import Image, ImageDraw

thumb_width, thumb_height = 160, 120
sheet = Image.new("RGB", (3 * thumb_width, 2 * (thumb_height + 22)), "white")
draw = ImageDraw.Draw(sheet)
for index, label in enumerate(class_names):
    sample = next(row for row in rows_by_split["train"] if row["label"] == label)
    x = index % 3 * thumb_width
    y = index // 3 * (thumb_height + 22)
    with Image.open(image_root / sample["path"]) as image:
        thumbnail = image.convert("RGB").resize((thumb_width, thumb_height), Image.Resampling.LANCZOS)
        sheet.paste(thumbnail, (x, y))
    draw.text((x + 6, y + thumb_height + 4), label, fill="black")
display(sheet)

## Next · Train the classifier

Build the six-logit head from Lab02 and train on `rows_by_split["train"]` with cross-entropy. Select the checkpoint on `rows_by_split["val"]` and save the class order and preprocessing parameters with it.